In [ ]:
import sqlite3
import polars as pl

from myutils import print_special, print_table
conn = sqlite3.connect("jobs.db")

pl.Config.set_fmt_str_lengths(20)
pl.Config.set_tbl_rows(200)

query = """
SELECT jobs.*, c.size
FROM jobs
left join companies c
  on jobs.company = c.name
WHERE remote = 1
  AND raw_text LIKE '%data%'
  and location like '%us%'
ORDER BY posted_at DESC;
"""

companies = """
SELECT *
FROM companies
"""

companies = pl.read_database(companies, conn, infer_schema_length=100000)

data = pl.read_database(query, conn)

data = data.with_columns(
    pl.col('posted_at').str.to_datetime(time_zone='UTC')
)
data = data.filter(pl.col('posted_at') > pl.datetime(2026, 3, 1, time_zone='UTC'))

data = data.filter(pl.col('company').str.contains(''))
#data = data.filter(pl.col('size').is_null())
data = data.top_k(k=10, by='score')

print_table(data.select(['company','url','posted_at', 'size','matched_keywords','score','notes']))


In [ ]:
companies = """
SELECT *
FROM companies
"""

companies = pl.read_database(companies, conn, infer_schema_length=10000)

display(companies)





In [ ]:
import sqlite3
import polars as pl

from myutils import print_special, print_table
conn = sqlite3.connect("jobs.db")

pl.Config.set_fmt_str_lengths(20)
pl.Config.set_tbl_rows(200)

query = """
delete jobs
FROM jobs
where posted_at is null
ORDER BY posted_at DESC;
"""

data = pl.read_database(query, conn, infer_schema_length=100000)

print(len(data))


174
